In [1]:
import subprocess
import sys
import os
import json
import requests
import numpy as np
import pandas as pd

print("Installing system dependencies...")
subprocess.run(["apt-get", "install", "-y", "-q", "openslide-tools"],
               check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "openslide-python", "-q"],
               check=True, capture_output=True)

import openslide
print(f"openslide: {openslide.__version__} ✓")

OUTPUT_DIR = "/kaggle/working/output"
SVS_DIR = "/kaggle/working/svs"
PATCHES_DIR = "/kaggle/working/patches/D8/BLOCKS_NORM_MACENKO"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SVS_DIR, exist_ok=True)
os.makedirs(PATCHES_DIR, exist_ok=True)

SUBTYPE_MAP = {
    "BRCA_LumA": "LumA", "BRCA_LumB": "LumB",
    "BRCA_Her2": "Her2", "BRCA_Basal": "Basal"
}

clinical_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "BLOCKS_NORM_MACENKO"]
    for f in files:
        if f == "data_clinical_patient.txt":
            clinical_path = os.path.join(root, f)
            break
    if clinical_path:
        break

print(f"Clinical file: {clinical_path}")
assert clinical_path is not None, "Clinical file not found"

df_clinical = pd.read_csv(clinical_path, sep="\t", comment="#", low_memory=False)

d8 = df_clinical[df_clinical["PATIENT_ID"].str.startswith("TCGA-D8-")].copy()
d8["subtype_mapped"] = d8["SUBTYPE"].map(SUBTYPE_MAP)
d8_valid = d8[d8["subtype_mapped"].notna()].copy()

target_subtypes = ["Her2", "LumB"]
d8_target = d8_valid[d8_valid["subtype_mapped"].isin(target_subtypes)]
target_patient_ids = d8_target["PATIENT_ID"].tolist()

print(f"D8 target patients: {len(target_patient_ids)}")
print(f"  Her2: {(d8_target['subtype_mapped']=='Her2').sum()}")
print(f"  LumB: {(d8_target['subtype_mapped']=='LumB').sum()}")

print("\nQuerying GDC API...")
endpoint = "https://api.gdc.cancer.gov/files"
filters = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.submitter_id", "value": target_patient_ids}},
        {"op": "=", "content": {"field": "data_type", "value": "Slide Image"}},
        {"op": "=", "content": {"field": "data_format", "value": "SVS"}},
        {"op": "like", "content": {"field": "file_name", "value": "%DX1%"}}
    ]
}
params = {
    "filters": json.dumps(filters),
    "fields": "file_id,file_name,cases.submitter_id,file_size",
    "format": "JSON",
    "size": 500
}
response = requests.get(endpoint, params=params)
hits = response.json()["data"]["hits"]

manifest = []
for hit in hits:
    case = hit.get("cases", [{}])[0].get("submitter_id", "unknown")
    subtype_vals = d8_target[d8_target["PATIENT_ID"] == case]["subtype_mapped"].values
    if len(subtype_vals) == 0:
        continue
    manifest.append({
        "file_id": hit["file_id"],
        "file_name": hit["file_name"],
        "patient_id": case,
        "subtype": subtype_vals[0],
        "file_size_gb": hit.get("file_size", 0) / 1e9
    })

manifest_df = pd.DataFrame(manifest)
print(f"Found {len(manifest_df)} DX1 slides")
print(manifest_df.groupby("subtype")[["file_id"]].count())
print(f"Total estimated size: {manifest_df['file_size_gb'].sum():.1f} GB")

def macenko_normalize(img, Io=240, alpha=1, beta=0.15):
    HERef = np.array([[0.5626, 0.2159],
                      [0.7201, 0.8012],
                      [0.4062, 0.5581]])
    maxCRef = np.array([1.9705, 1.0308])
    img = img.astype(np.float64)
    img = np.clip(img, 1, 255)
    OD = -np.log(img / Io)
    ODhat = OD.reshape(-1, 3)
    ODhat = ODhat[~np.any(ODhat < beta, axis=1)]
    if len(ODhat) < 10:
        return img.astype(np.uint8)
    _, _, V = np.linalg.svd(ODhat, full_matrices=False)
    V = V[:2].T
    That = ODhat @ V
    phi = np.arctan2(That[:, 1], That[:, 0])
    minPhi = np.percentile(phi, alpha)
    maxPhi = np.percentile(phi, 100 - alpha)
    vMin = V @ np.array([np.cos(minPhi), np.sin(minPhi)])
    vMax = V @ np.array([np.cos(maxPhi), np.sin(maxPhi)])
    if vMin[0] > vMax[0]:
        HE = np.array([vMin, vMax]).T
    else:
        HE = np.array([vMax, vMin]).T
    OD_flat = OD.reshape(-1, 3)
    C = np.linalg.lstsq(HE, OD_flat.T, rcond=None)[0]
    maxC = np.percentile(C, 99, axis=1)
    maxC = np.where(maxC == 0, 1e-6, maxC)
    C = C / maxC[:, np.newaxis] * maxCRef[:, np.newaxis]
    with np.errstate(over="ignore"):
        Inorm = Io * np.exp(-HERef @ C)
    Inorm = np.clip(Inorm, 0, 255).T.reshape(img.shape).astype(np.uint8)
    return Inorm

def is_tissue_patch(img, threshold=0.5, sat_threshold=20):
    from skimage.color import rgb2hsv
    hsv = rgb2hsv(img)
    return np.mean(hsv[:, :, 1] > sat_threshold / 255.0) > threshold

def extract_patches_from_svs(svs_path, patient_id, subtype,
                              patches_base_dir, patch_size=256,
                              target_mpp=1.0, max_patches=2000):
    from PIL import Image as PILImage
    slide = openslide.OpenSlide(svs_path)
    mpp_x = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_X, 0.5))
    level = slide.get_best_level_for_downsample(mpp_x / target_mpp)
    level_downsample = slide.level_downsamples[level]
    actual_mpp = mpp_x * level_downsample
    patch_size_level0 = int(patch_size * (target_mpp / mpp_x))
    w0, h0 = slide.dimensions
    patch_dir = os.path.join(patches_base_dir, patient_id)
    os.makedirs(patch_dir, exist_ok=True)
    coords = [(x, y)
              for x in range(0, w0 - patch_size_level0, patch_size_level0)
              for y in range(0, h0 - patch_size_level0, patch_size_level0)]
    np.random.seed(42)
    np.random.shuffle(coords)
    patches_meta = []
    extracted = 0
    skipped_bg = 0
    for x, y in coords:
        if extracted >= max_patches:
            break
        patch = slide.read_region((x, y), level, (patch_size, patch_size))
        patch_np = np.array(patch.convert("RGB"))
        if not is_tissue_patch(patch_np):
            skipped_bg += 1
            continue
        try:
            patch_norm = macenko_normalize(patch_np)
        except Exception:
            patch_norm = patch_np
        fname = f"{patient_id}_({x},{y}).jpg"
        fpath = os.path.join(patch_dir, fname)
        PILImage.fromarray(patch_norm).save(fpath, quality=95)
        patches_meta.append({
            "patient_id": patient_id,
            "subtype_clean": subtype,
            "institution": "D8",
            "patch_path": fpath,
            "x": x, "y": y,
            "actual_mpp": round(actual_mpp, 4)
        })
        extracted += 1
    slide.close()
    print(f"  {patient_id} ({subtype}): {extracted} patches, {skipped_bg} background skipped")
    return patches_meta

def download_svs(file_id, file_name, dest_dir):
    url = f"https://api.gdc.cancer.gov/data/{file_id}"
    dest_path = os.path.join(dest_dir, file_name)
    response = requests.get(url, stream=True, timeout=3600)
    response.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)
    return dest_path

all_patches_meta = []
failed_files = []

print(f"\nStarting: {len(manifest_df)} D8 slides")

for idx, row in manifest_df.iterrows():
    file_id = row["file_id"]
    file_name = row["file_name"]
    patient_id = row["patient_id"]
    subtype = row["subtype"]
    size_gb = row["file_size_gb"]

    print(f"\n[{idx+1}/{len(manifest_df)}] {patient_id} ({subtype}) | {size_gb:.2f} GB")
    statvfs = os.statvfs("/kaggle/working")
    free_gb = (statvfs.f_frsize * statvfs.f_bavail) / 1e9
    print(f"  Free disk: {free_gb:.1f} GB")

    if free_gb < size_gb + 1.5:
        print(f"  SKIPPING — insufficient disk space")
        failed_files.append({"file_id": file_id, "patient_id": patient_id,
                             "subtype": subtype, "reason": "disk_full"})
        continue

    try:
        print(f"  Downloading...")
        svs_path = download_svs(file_id, file_name, SVS_DIR)
        print(f"  Downloaded: {os.path.getsize(svs_path)/1e9:.2f} GB")
    except Exception as e:
        print(f"  DOWNLOAD FAILED: {e}")
        failed_files.append({"file_id": file_id, "patient_id": patient_id,
                             "subtype": subtype, "reason": f"download_failed: {e}"})
        continue

    try:
        meta = extract_patches_from_svs(
            svs_path, patient_id, subtype,
            PATCHES_DIR, patch_size=256, target_mpp=1.0, max_patches=2000
        )
        all_patches_meta.extend(meta)
    except Exception as e:
        print(f"  EXTRACTION FAILED: {e}")
        failed_files.append({"file_id": file_id, "patient_id": patient_id,
                             "subtype": subtype, "reason": f"extraction_error: {e}"})

    os.remove(svs_path)
    print(f"  SVS deleted. Total patches so far: {len(all_patches_meta)}")

patches_df = pd.DataFrame(all_patches_meta)
print(f"\nTotal D8 patches extracted: {len(patches_df)}")
if len(patches_df) > 0:
    print(patches_df["subtype_clean"].value_counts())

patches_df.to_csv(os.path.join(OUTPUT_DIR, "d8_her2_lumb_patches.csv"), index=False)

if failed_files:
    pd.DataFrame(failed_files).to_csv(
        os.path.join(OUTPUT_DIR, "d8_failed_files.csv"), index=False)
    print(f"Failed: {len(failed_files)} — see d8_failed_files.csv")

print(f"\nOutputs in {OUTPUT_DIR}/:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  - {f} ({size:.1f} MB)")

print("\nDone. Next: create Dataset bc-xai-d8-patches → merge all CSVs → retrain v3.")

Installing system dependencies...
openslide: 1.4.6 ✓
Clinical file: /kaggle/input/datasets/anishapanja/brca-pancancer-atlas-clinical/data_clinical_patient.txt
D8 target patients: 28
  Her2: 6
  LumB: 22

Querying GDC API...
Found 80 DX1 slides
         file_id
subtype         
Her2          13
LumB          67
Total estimated size: 78.3 GB

Starting: 80 D8 slides

[1/80] TCGA-D8-A1JG (Her2) | 0.17 GB
  Free disk: 20.9 GB
  Downloading...
  Downloaded: 0.17 GB
  TCGA-D8-A1JG (Her2): 294 patches, 1677 background skipped
  SVS deleted. Total patches so far: 294

[2/80] TCGA-D8-A1Y2 (LumB) | 1.15 GB
  Free disk: 20.9 GB
  Downloading...
  Downloaded: 1.15 GB
  TCGA-D8-A1Y2 (LumB): 2000 patches, 3705 background skipped
  SVS deleted. Total patches so far: 2294

[3/80] TCGA-D8-A1Y3 (LumB) | 1.94 GB
  Free disk: 20.9 GB
  Downloading...
  Downloaded: 1.94 GB
  TCGA-D8-A1Y3 (LumB): 2000 patches, 1724 background skipped
  SVS deleted. Total patches so far: 4294

[4/80] TCGA-D8-A1X6 (LumB) | 1.4